In [ ]:
import pandas as pd
import re
from pathlib import Path

In [54]:
main_file = Path(r"D:\Proyek FEB\lengkapin data\checkpoint_2_merged_scimago_database_q.xlsx")
scimago_file = Path(r"D:\Proyek FEB\scimagojr 2025.csv")

df = pd.read_excel(main_file)

scimago = pd.read_csv(
    scimago_file,
    sep=";",
    quotechar='"',
    encoding="utf-8"
)

COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

SCIMAGO_TITLE = "Title"
SCIMAGO_Q = "SJR Best Quartile"
SCIMAGO_ISSN = "Issn"
SCIMAGO_PUBLISHER = "Publisher"
SCIMAGO_COVERAGE = "Coverage"
SCIMAGO_CATEGORIES = "Categories"

In [55]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "??",
        "???",
        "n/a",
        "na",
        "tidak ada",
        "not found"
    ]:
        return True
    
    return False


def raw_title_key(value):
    """
    Raw title matching:
    - lowercase
    - trim
    - rapikan spasi
    - karakter penting tetap dipertahankan
    
    Contoh:
    CA-A Cancer Journal for Clinicians
    menjadi:
    ca-a cancer journal for clinicians
    
    Bukan:
    caa cancer journal for clinicians
    """
    if pd.isna(value):
        return ""
    
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    
    return text


def clean_q_from_scimago(value):
    """
    Aturan:
    - Jika ada Q1/Q2/Q3/Q4, ambil Q tersebut.
    - Jika jurnal ada di database tapi kolom Q kosong/tidak memuat Q, isi no-Q.
    """
    if pd.isna(value):
        return "no-Q"
    
    text = str(value).strip()
    
    if text == "":
        return "no-Q"
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "n/a",
        "na"
    ]:
        return "no-Q"
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return f"Q{match.group(1)}"
    
    return "no-Q"

In [56]:
scimago_clean = scimago.copy()

scimago_clean["raw_key"] = scimago_clean[SCIMAGO_TITLE].apply(raw_title_key)
scimago_clean["scimago_q_clean"] = scimago_clean[SCIMAGO_Q].apply(clean_q_from_scimago)

scimago_clean[
    [
        SCIMAGO_TITLE,
        "raw_key",
        SCIMAGO_Q,
        "scimago_q_clean",
        SCIMAGO_ISSN,
        SCIMAGO_PUBLISHER,
        SCIMAGO_COVERAGE,
        SCIMAGO_CATEGORIES
    ]
].head()

,Title,raw_key,SJR Best Quartile,scimago_q_clean,Issn,Publisher,Coverage,Categories
0,Ca-A Cancer Journal for Clinicians,ca-a cancer journal for clinicians,Q1,Q1,"15424863, 00079235",John Wiley and Sons Inc,1950-2026,Hematology (Q1); Oncology (Q1)
1,Nature Reviews Molecular Cell Biology,nature reviews molecular cell biology,Q1,Q1,"14710072, 14710080",Nature Research,2000-2026,Cell Biology (Q1); Molecular Biology (Q1)
2,Quarterly Journal of Economics,quarterly journal of economics,Q1,Q1,"00335533, 15314650",Oxford University Press,1886-2026,Economics and Econometrics (Q1)
3,Nature Reviews Drug Discovery,nature reviews drug discovery,Q1,Q1,"14741784, 14741776",Nature Research,2002-2026,Drug Discovery (Q1); Medicine (miscellaneous) ...
4,Nature Reviews Clinical Oncology,nature reviews clinical oncology,Q1,Q1,"17594782, 17594774",Nature Publishing Group,2009-2026,Oncology (Q1)


In [57]:
raw_q_count = (
    scimago_clean
    .dropna(subset=["raw_key"])
    .groupby("raw_key")["scimago_q_clean"]
    .nunique()
    .reset_index(name="unique_q_count")
)

raw_conflict = raw_q_count[raw_q_count["unique_q_count"] > 1].copy()

raw_conflict_keys = set(raw_conflict["raw_key"])

print("Jumlah raw title conflict di database:", len(raw_conflict_keys))

Jumlah raw title conflict di database: 10


In [58]:
safe_scimago = scimago_clean[
    ~scimago_clean["raw_key"].isin(raw_conflict_keys)
].copy()

raw_q_map = (
    safe_scimago
    .dropna(subset=["raw_key"])
    .drop_duplicates(subset=["raw_key"], keep="first")
    .set_index("raw_key")["scimago_q_clean"]
    .to_dict()
)

print("Jumlah jurnal aman untuk mapping:", len(raw_q_map))

Jumlah jurnal aman untuk mapping: 32171


In [59]:
df_checkpoint2 = df.copy()

df_checkpoint2["raw_key"] = df_checkpoint2[COL_JOURNAL].apply(raw_title_key)
df_checkpoint2["q_source_checkpoint2"] = ""

updated_count = 0
not_found_count = 0
conflict_count = 0
same_value_count = 0

for idx, row in df_checkpoint2.iterrows():
    raw_key = row["raw_key"]
    
    # Jika konflik di database SCImago, jangan overwrite
    if raw_key in raw_conflict_keys:
        conflict_count += 1
        df_checkpoint2.at[idx, "q_source_checkpoint2"] = "conflict_in_scimago_database_keep_existing"
        continue
    
    # Jika jurnal ditemukan di database SCImago,
    # SELALU pakai Q dari SCImago, meskipun Q lama sudah terisi
    if raw_key in raw_q_map:
        new_q = raw_q_map[raw_key]
        old_q = df_checkpoint2.at[idx, COL_Q]
        
        df_checkpoint2.at[idx, COL_Q] = new_q
        df_checkpoint2.at[idx, "q_source_checkpoint2"] = "scimago_2025_database"
        
        if old_q != new_q:
            updated_count += 1
        else:
            same_value_count += 1
    
    # Jika tidak ditemukan di database, biarkan nilai lama
    else:
        not_found_count += 1
        df_checkpoint2.at[idx, "q_source_checkpoint2"] = "not_found_in_scimago_keep_existing"

print("Jumlah baris Q yang diganti/disamakan dari database SCImago:", updated_count)
print("Jumlah baris yang sudah sama dengan database SCImago:", same_value_count)
print("Jumlah baris jurnal tidak ditemukan di database SCImago:", not_found_count)
print("Jumlah baris masuk konflik raw title:", conflict_count)

Jumlah baris Q yang diganti/disamakan dari database SCImago: 184
Jumlah baris yang sudah sama dengan database SCImago: 1180
Jumlah baris jurnal tidak ditemukan di database SCImago: 789
Jumlah baris masuk konflik raw title: 0


In [60]:
df_checkpoint2[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      571
Q2      558
?       432
Q3      252
no-Q    209
Q4      131
Name: count, dtype: int64

In [61]:
remaining_missing_q = df_checkpoint2[
    df_checkpoint2[COL_Q].apply(is_empty_value)
].copy()

print("Sisa baris SCOPUS Q masih kosong / ?:", len(remaining_missing_q))

Sisa baris SCOPUS Q masih kosong / ?: 432


In [62]:
remaining_missing_unique = (
    remaining_missing_q[[COL_JOURNAL, "raw_key", "q_source_checkpoint2"]]
    .drop_duplicates()
    .sort_values(COL_JOURNAL)
    .reset_index(drop=True)
)

print("Sisa jurnal unik yang belum match:", len(remaining_missing_unique))

remaining_missing_unique.head(50)

Sisa jurnal unik yang belum match: 88


,Journal,raw_key,q_source_checkpoint2
0,- Ilkogretim Online - Elementary Education Onl...,- ilkogretim online - elementary education onl...,not_found_in_scimago_keep_existing
1,"- International Journal of Innovation, Creativ...","- international journal of innovation, creativ...",not_found_in_scimago_keep_existing
2,- Journal of Islamic Marketing\n- Studies in S...,- journal of islamic marketing - studies in sy...,not_found_in_scimago_keep_existing
3,- Review of International Geographical Educati...,- review of international geographical educati...,not_found_in_scimago_keep_existing
4,- Review of International Geographical Educati...,- review of international geographical educati...,not_found_in_scimago_keep_existing
5,- Review of International Geographical Educati...,- review of international geographical educati...,not_found_in_scimago_keep_existing
6,--,--,not_found_in_scimago_keep_existing
7,2024 International Conference on Sustainable I...,2024 international conference on sustainable i...,not_found_in_scimago_keep_existing
8,2025 International Conference on Sustainable I...,2025 international conference on sustainable i...,not_found_in_scimago_keep_existing
9,AL-IHKAM Jurnal Hukum & Pranata Sosial,al-ihkam jurnal hukum & pranata sosial,not_found_in_scimago_keep_existing


In [63]:
remaining_missing_unique.to_excel("nama jurnal yang tidak ada di database scimago.xlsx", index=False)

In [64]:
conflict_rows_in_main = df_checkpoint2[
    df_checkpoint2["raw_key"].isin(raw_conflict_keys)
].copy()

print("Jumlah baris data utama yang terkena konflik database:", len(conflict_rows_in_main))

Jumlah baris data utama yang terkena konflik database: 0


In [65]:
output_file_with_log_col = "sudah 40% pakai ini.xlsx"

df_checkpoint2.to_excel(output_file_with_log_col, index=False)

print(f"Saved: {output_file_with_log_col}")

Saved: sudah 40% pakai ini.xlsx
